In [38]:
import torch
from torch.distributions import Normal

mean, std = torch.tensor([1., 2., 4.]), torch.tensor([0.1, 0.5, 2])
normal_distr = Normal(mean, std)
action = normal_distr.rsample()
log_prob = normal_distr.log_prob(action)
log_prob.sum(0, keepdim=True)

tensor([-1.6393])

In [24]:
import torch
import torch.optim as optim
from torch.distributions import Normal
import torch.nn as nn

class NeuralNetwork(nn.Module):
    '''
    This class implements a neural network with a variable number of hidden layers and hidden units.
    You may use this function to parametrize your policy and critic networks.
    '''
    def __init__(self, input_dim: int, output_dim: int, hidden_size: int, 
                                hidden_layers: int, activation: str):
        super(NeuralNetwork, self).__init__()

        # TODO: Implement this function which should define a neural network 
        # with a variable number of hidden layers and hidden units.
        # Here you should define layers which your network will use.
        self.input = nn.Linear(input_dim, hidden_size)        
        self.hidden = nn.Sequential()
        for i in range(hidden_layers):
            self.hidden.append(nn.Linear(hidden_size, hidden_size))
        self.output = nn.Linear(hidden_size, output_dim)
        
        activation_dict = {
            'relu': nn.ReLU(),
            'sigmoid': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }
        if activation not in activation_dict.keys():
            raise KeyError(f'The activation "{activation}" is unknown.')
        self.activation_fct = activation_dict[activation]
        
    def forward(self, s: torch.Tensor) -> torch.Tensor:
        # TODO: Implement the forward pass for the neural network you have defined.
        s = self.input(self.activation_fct(s))
        s = self.hidden(self.activation_fct(s))
        s = self.output(self.activation_fct(s))
        
        return s

In [25]:
mynetwork = NeuralNetwork(15, 5, 30, 4, 'relu')
print(mynetwork)

NeuralNetwork(
  (input): Linear(in_features=15, out_features=30, bias=True)
  (hidden): Sequential(
    (0): Linear(in_features=30, out_features=30, bias=True)
    (1): Linear(in_features=30, out_features=30, bias=True)
    (2): Linear(in_features=30, out_features=30, bias=True)
    (3): Linear(in_features=30, out_features=30, bias=True)
  )
  (output): Linear(in_features=30, out_features=5, bias=True)
  (activation_fct): ReLU()
)


In [3]:
import numpy as np

np.random.uniform(0, 1)

0.5889683117875393

In [18]:
import torch
import torch.nn as nn

class NeuralNetwork(nn.Module):
    '''
    This class implements a neural network with a variable number of hidden layers and hidden units.
    You may use this function to parametrize your policy and critic networks.
    '''
    def __init__(self, input_dim: int, output_dim: int, hidden_size: int, 
                                hidden_layers: int, activation: str):
        super(NeuralNetwork, self).__init__()

        # TODO: Implement this function which should define a neural network 
        # with a variable number of hidden layers and hidden units.
        # Here you should define layers which your network will use.
        activation_dict = {
            'relu': nn.ReLU(),
            'sigmoid': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }
        if activation not in activation_dict.keys():
            raise KeyError(f'The activation "{activation}" is unknown.')
        self.activation_fct = activation_dict[activation]
        
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.BatchNorm1d(hidden_size),
            self.activation_fct
        )
        for _ in range(hidden_layers):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
            self.layers.append(nn.BatchNorm1d(hidden_size))
            self.layers.append(self.activation_fct)
            
        self.mean = nn.Linear(hidden_size, output_dim)
        self.log_std = nn.Linear(hidden_size, output_dim)
        
    def forward(self, s: torch.Tensor) -> torch.Tensor:
        # TODO: Implement the forward pass for the neural network you have defined.
        s = self.layers(s.unsqueeze(0))
        mean = self.mean(s)
        log_std = self.log_std(s)
        return mean, log_std

In [20]:
net = NeuralNetwork(input_dim=3, output_dim=1, hidden_layers=2, hidden_size=64, activation='relu')
net(torch.tensor([[1., 2., 3.],
                  [1., 2., 4.]]))

RuntimeError: running_mean should contain 2 elements not 64